# 🍽️ Zomato Restaurant Clustering & Sentiment Analysis
**Unsupervised ML project (Hyderabad restaurants)**

## 1. Project Summary
**Business context.** Zomato lists restaurants, menus and customer reviews. Customers want to find the *best restaurant* in their locality, and Zomato wants to know which kinds of restaurants it has and where it is lagging.

**Goals**
1. Clean and explore the restaurant and review data (EDA + visualizations).
2. Analyse the **sentiment** of customer reviews.
3. **Cluster** the restaurants into meaningful segments.
4. Turn each segment into a business strategy for customers and for Zomato.

**Data**
| File | Rows | What it holds |
|---|---|---|
| `Zomato_Restaurant_names_and_Metadata.csv` | 105 | Name, Links, Cost, Collections, Cuisines, Timings |
| `Zomato_Restaurant_reviews.csv` | 10,000 | Restaurant, Reviewer, Review, Rating, Metadata (reviews/followers), Time, Pictures |

**Pipeline:** EDA → Cleaning → Feature engineering → Scaling → Clustering → Explainability → (extra) Review-sentiment classifier.

## Table of contents
1. Project summary · 2. Setup · 3. Load data · 4. Data cleaning · 5. EDA & visualization · 6. Hypotheses · 7. Sentiment analysis · 8. Feature engineering · 9. Clustering · 10. Cluster profiles & strategies · 11. Predicting review sentiment (model + metrics) · 12. Conclusion

## 2. Setup

In [ ]:
# VADER = a ready-made sentiment analyser for short, informal text (works well for reviews).
# Safe to re-run. In Colab the '!' runs a shell command.
!pip install -q vaderSentiment

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, mannwhitneyu
from scipy.cluster.hierarchy import linkage, dendrogram

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
RANDOM_STATE = 42   # fixed so results are reproducible

## 3. Load the data
If the CSVs are not in the Colab session yet, you will be asked to upload them.

In [ ]:
REVIEWS_PATH = "Zomato_Restaurant_reviews.csv"
META_PATH = "Zomato_Restaurant_names_and_Metadata.csv"

if not (os.path.exists(REVIEWS_PATH) and os.path.exists(META_PATH)):
    try:
        from google.colab import files
        print("Upload BOTH csv files:")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put both CSV files next to this notebook.")

reviews_raw = pd.read_csv(REVIEWS_PATH)
meta_raw = pd.read_csv(META_PATH)

print("Reviews :", reviews_raw.shape)
print("Metadata:", meta_raw.shape)

In [ ]:
display(meta_raw.head())
display(reviews_raw.head())

In [ ]:
# Quick data-quality overview
for name, df in [("Restaurant metadata", meta_raw), ("Reviews", reviews_raw)]:
    print(f"===== {name} =====")
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values per column:")
    print(df.isna().sum()[df.isna().sum() > 0].to_string() or "none")
    print()

print("Odd Rating values:", sorted(set(reviews_raw['Rating'].dropna()) - set(map(str, [1, 2, 3, 4, 5, 1.5, 2.5, 3.5, 4.5]))))

**What I notice (before cleaning)**
- `Cost` is text with commas (e.g. `"1,300"`), so it must be converted to a number.
- `Rating` is text and has a stray value (`"Like"`).
- `Metadata` packs two numbers into one string (`"96 Reviews , 1069 Followers"`).
- Reviews has duplicate rows and ~38 rows that are almost entirely empty.
- `Collections` is missing for about half of the restaurants (many simply have no collection tag).

## 4. Data cleaning
Cleaning is wrapped in functions so the steps are reusable and easy to read.

In [ ]:
def clean_reviews(df):
    """Return a cleaned copy of the reviews table."""
    df = df.copy()
    df["Restaurant"] = df["Restaurant"].str.strip()

    # 1) remove exact duplicate rows
    df = df.drop_duplicates()

    # 2) Rating -> numeric ("Like" and other junk become NaN), then drop rows without a rating
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
    df = df.dropna(subset=["Rating"])

    # 3) split Metadata "96 Reviews , 1069 Followers" into two numeric columns
    meta_txt = df["Metadata"].fillna("").str.replace(",", "", regex=False)
    df["reviewer_reviews"] = pd.to_numeric(meta_txt.str.extract(r"(\d+)\s*Review")[0], errors="coerce").fillna(0)
    df["reviewer_followers"] = pd.to_numeric(meta_txt.str.extract(r"(\d+)\s*Follower")[0], errors="coerce").fillna(0)

    # 4) time features
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df["year"] = df["Time"].dt.year

    # 5) review text: fill missing with "", measure length in words
    df["Review"] = df["Review"].fillna("").astype(str).str.strip()
    df["review_len"] = df["Review"].str.split().str.len()

    return df.drop(columns=["Metadata"]).reset_index(drop=True)


def clean_metadata(df):
    """Return a cleaned copy of the restaurant table."""
    df = df.copy()
    df["Name"] = df["Name"].str.strip()
    df = df.drop_duplicates(subset="Name")

    # Cost: "1,300" -> 1300.0
    df["Cost"] = pd.to_numeric(df["Cost"].astype(str).str.replace(",", "", regex=False), errors="coerce")

    # Missing collections/cuisines -> empty string (no collection tag is a real, valid state)
    df["Collections"] = df["Collections"].fillna("")
    df["Cuisines"] = df["Cuisines"].fillna("")

    # List-type columns -> counts
    df["cuisine_list"] = df["Cuisines"].apply(lambda s: [c.strip() for c in s.split(",") if c.strip()])
    df["n_cuisines"] = df["cuisine_list"].str.len()
    df["n_collections"] = df["Collections"].apply(lambda s: len([c for c in s.split(",") if c.strip()]))
    return df.reset_index(drop=True)


reviews = clean_reviews(reviews_raw)
meta = clean_metadata(meta_raw)

print("Reviews after cleaning :", reviews.shape, "(was", reviews_raw.shape, ")")
print("Metadata after cleaning:", meta.shape)
print("Missing Cost values    :", meta["Cost"].isna().sum())
display(reviews.head(3))

In [ ]:
# Some rows have a rating but no review text. Keep them for rating analysis,
# but use only rows WITH text for sentiment analysis.
reviews_text = reviews[reviews["review_len"] > 0].copy()
print("Rows with review text:", len(reviews_text), "of", len(reviews))

# Do the two files describe the same restaurants?
in_both = set(reviews["Restaurant"]) & set(meta["Name"])
print("Restaurants in both files:", len(in_both))
print("Only in metadata (no reviews):", sorted(set(meta["Name"]) - set(reviews["Restaurant"])))

## 5. EDA & visualization

### 5.1 Rating distribution

In [ ]:
order = sorted(reviews["Rating"].unique())
ax = sns.countplot(data=reviews, x="Rating", order=order)
ax.set_title("Distribution of review ratings")
ax.set_xlabel("Rating (stars)"); ax.set_ylabel("Number of reviews")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.show()
print((reviews["Rating"].value_counts(normalize=True).sort_index() * 100).round(1).to_string())

**Observation:** ratings are skewed towards the top (5 and 4 stars dominate), so the data is imbalanced. Keep this in mind for the sentiment classifier (section 11).

### 5.2 Cost distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(meta["Cost"], bins=20, kde=True, ax=axes[0])
axes[0].set_title("Cost for one person (₹)"); axes[0].set_xlabel("Cost")
sns.boxplot(x=meta["Cost"], ax=axes[1])
axes[1].set_title("Cost – outlier check")
plt.tight_layout(); plt.show()
print(meta["Cost"].describe().round(0).to_string())

### 5.3 Most common cuisines & cost by cuisine

In [ ]:
# One row per (restaurant, cuisine)
cuisine_long = meta[["Name", "Cost", "cuisine_list"]].explode("cuisine_list").rename(columns={"cuisine_list": "Cuisine"})
cuisine_counts = cuisine_long["Cuisine"].value_counts()

plt.figure(figsize=(9, 5))
sns.barplot(x=cuisine_counts.head(12).values, y=cuisine_counts.head(12).index, color="#4C72B0")
plt.title("Top 12 cuisines (number of restaurants serving them)")
plt.xlabel("Number of restaurants"); plt.ylabel("")
plt.show()

top10 = cuisine_counts.head(10).index
plt.figure(figsize=(10, 5))
sns.boxplot(data=cuisine_long[cuisine_long["Cuisine"].isin(top10)], x="Cost", y="Cuisine", order=top10)
plt.title("Cost distribution across the 10 most common cuisines")
plt.show()

### 5.4 Reviews over time

In [ ]:
monthly = reviews["Time"].dt.to_period("M").value_counts().sort_index()
monthly.index = monthly.index.to_timestamp()
ax = monthly.plot(figsize=(11, 4))
ax.set_title("Number of reviews per month"); ax.set_xlabel(""); ax.set_ylabel("Reviews")
plt.show()

### 5.5 Which restaurants are most popular? (highest average rating)

In [ ]:
rest_rating = (reviews.groupby("Restaurant")["Rating"].agg(avg_rating="mean", n="size")
               .sort_values("avg_rating", ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=rest_rating.head(10)["avg_rating"], y=rest_rating.head(10).index, color="#55A868", ax=axes[0])
axes[0].set_title("Top 10 restaurants by average rating"); axes[0].set_xlim(0, 5); axes[0].set_ylabel("")
sns.barplot(x=rest_rating.tail(10)["avg_rating"], y=rest_rating.tail(10).index, color="#C44E52", ax=axes[1])
axes[1].set_title("Bottom 10 restaurants by average rating"); axes[1].set_xlim(0, 5); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

### 5.6 Cost vs rating

In [ ]:
cost_rating = meta.merge(rest_rating.reset_index(), left_on="Name", right_on="Restaurant")
sns.scatterplot(data=cost_rating, x="Cost", y="avg_rating")
sns.regplot(data=cost_rating, x="Cost", y="avg_rating", scatter=False, color="red")
plt.title("Cost for one vs average rating (one dot = one restaurant)")
plt.show()

### 5.7 Reviewers: who are the critics?

In [ ]:
reviewers = (reviews.groupby("Reviewer")
             .agg(reviewer_reviews=("reviewer_reviews", "max"),
                  reviewer_followers=("reviewer_followers", "max"),
                  avg_rating_given=("Rating", "mean"),
                  reviews_in_data=("Rating", "size"))
             .reset_index())

print("Top 10 reviewers by followers:")
display(reviewers.nlargest(10, "reviewer_followers"))

# Define a "critic" as someone in the top 5% by followers AND with 10+ reviews
follower_cut = reviewers["reviewer_followers"].quantile(0.95)
critics = reviewers[(reviewers["reviewer_followers"] >= follower_cut) & (reviewers["reviewer_reviews"] >= 10)]
print(f"\nCritic threshold: >= {follower_cut:.0f} followers and >= 10 reviews -> {len(critics)} critics")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(np.log1p(reviewers["reviewer_followers"]), bins=30, ax=axes[0])
axes[0].set_title("Reviewer followers (log scale)"); axes[0].set_xlabel("log(1 + followers)")
sns.histplot(reviews["Pictures"], bins=15, ax=axes[1])
axes[1].set_title("Pictures posted per review")
plt.tight_layout(); plt.show()

**Observation:** most reviewers have very few followers while a small group has a large following. The follower counts are heavily skewed, which is why we use a log transform later.

## 6. Hypotheses from the visualizations
We turn what we saw into testable statements. All three use non-parametric tests, because ratings and followers are not normally distributed.

- **H1:** More expensive restaurants get higher ratings.
- **H2:** Reviews that include pictures give higher ratings.
- **H3:** "Critics" (reviewers with many followers) give lower ratings than other reviewers.

In [ ]:
# H1: cost vs average rating (restaurant level)
rho, p = spearmanr(cost_rating["Cost"], cost_rating["avg_rating"])
print(f"H1  Cost vs avg rating   : Spearman rho = {rho:.3f}, p-value = {p:.4f}")

# H2: reviews with pictures vs without
with_pic = reviews.loc[reviews["Pictures"] > 0, "Rating"]
no_pic = reviews.loc[reviews["Pictures"] == 0, "Rating"]
u, p = mannwhitneyu(with_pic, no_pic)
print(f"H2  With pictures: mean {with_pic.mean():.2f} (n={len(with_pic)}) | without: mean {no_pic.mean():.2f} (n={len(no_pic)}) | p = {p:.4f}")

# H3: critics vs everyone else (per review)
critic_names = set(critics["Reviewer"])
is_critic = reviews["Reviewer"].isin(critic_names)
u, p = mannwhitneyu(reviews.loc[is_critic, "Rating"], reviews.loc[~is_critic, "Rating"])
print(f"H3  Critics: mean {reviews.loc[is_critic, 'Rating'].mean():.2f} (n={is_critic.sum()}) | others: mean {reviews.loc[~is_critic, 'Rating'].mean():.2f} | p = {p:.4f}")
print("\n(p < 0.05 means the difference is statistically significant)")

**Conclusion of the hypothesis tests:** _(write 2–3 lines in your own words after you read the numbers above: which hypotheses are supported, which are not, and how strong the effect really is.)_

## 7. Sentiment analysis
We use **VADER**, which gives a *compound* score from -1 (very negative) to +1 (very positive) for each review.
Standard labels: `>= 0.05` Positive · `<= -0.05` Negative · otherwise Neutral.

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return sia.polarity_scores(text)["compound"]

def label_sentiment(score):
    if score >= 0.05:
        return "Positive"
    if score <= -0.05:
        return "Negative"
    return "Neutral"

reviews_text["sentiment"] = reviews_text["Review"].apply(get_sentiment)
reviews_text["sentiment_label"] = reviews_text["sentiment"].apply(label_sentiment)

print(reviews_text["sentiment_label"].value_counts(normalize=True).mul(100).round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=reviews_text, x="sentiment_label", order=["Positive", "Neutral", "Negative"], ax=axes[0])
axes[0].set_title("Sentiment of reviews")
sns.boxplot(data=reviews_text, x="Rating", y="sentiment", ax=axes[1])
axes[1].set_title("VADER sentiment score by star rating")
plt.tight_layout(); plt.show()

print("Correlation between star rating and sentiment score (Spearman):",
      round(spearmanr(reviews_text["Rating"], reviews_text["sentiment"])[0], 3))

In [ ]:
# Where do stars and text disagree? (e.g. 1-star rating but positive text)
reviews_text["rating_group"] = pd.cut(reviews_text["Rating"], bins=[0, 2.5, 3.5, 5], labels=["Low (1-2.5)", "Mid (3-3.5)", "High (4-5)"])
agreement = pd.crosstab(reviews_text["rating_group"], reviews_text["sentiment_label"], normalize="index").mul(100).round(1)
display(agreement)

**Observation:** _(describe how well text sentiment matches the star rating, and where they disagree.)_

## 8. Feature engineering
We want **one row per restaurant** that combines restaurant info (cost, cuisines, collections) with a summary of its reviews (rating, sentiment, engagement).

In [ ]:
def build_restaurant_features(rev, meta_df):
    """One row per restaurant = metadata + aggregated review statistics."""
    agg = rev.groupby("Restaurant").agg(
        avg_rating=("Rating", "mean"),
        avg_sentiment=("sentiment", "mean"),
        pct_negative=("sentiment_label", lambda s: (s == "Negative").mean() * 100),
        avg_pictures=("Pictures", "mean"),
        avg_review_words=("review_len", "mean"),
        avg_reviewer_followers=("reviewer_followers", "mean"),
        n_reviews=("Rating", "size"),
    ).reset_index()
    out = meta_df.merge(agg, left_on="Name", right_on="Restaurant", how="inner").drop(columns="Restaurant")
    return out


restaurants = build_restaurant_features(reviews_text, meta)

# Feature encoding: 0/1 flags for the most common cuisines
TOP_N_CUISINES = 8
top_cuisines = list(cuisine_counts.head(TOP_N_CUISINES).index)
for c in top_cuisines:
    restaurants["cuis_" + c.replace(" ", "_")] = restaurants["cuisine_list"].apply(lambda lst, c=c: int(c in lst))

# New features
restaurants["log_followers"] = np.log1p(restaurants["avg_reviewer_followers"])   # tame the skew
restaurants["has_collection"] = (restaurants["n_collections"] > 0).astype(int)

print("Restaurants used for clustering:", len(restaurants))
display(restaurants.drop(columns=["Links", "Collections", "Cuisines", "Timings", "cuisine_list"]).head())

In [ ]:
# Outlier check with the IQR rule on the numeric features we plan to cluster on
CLUSTER_FEATURES = ["Cost", "avg_rating", "avg_sentiment", "pct_negative",
                    "avg_pictures", "avg_review_words", "log_followers", "n_cuisines"]

def iqr_outlier_report(df, cols):
    rows = []
    for c in cols:
        q1, q3 = df[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        rows.append((c, int(((df[c] < lo) | (df[c] > hi)).sum()), round(lo, 2), round(hi, 2)))
    return pd.DataFrame(rows, columns=["feature", "n_outliers", "lower_bound", "upper_bound"])

display(iqr_outlier_report(restaurants, CLUSTER_FEATURES))
print("Missing values in clustering features:\n", restaurants[CLUSTER_FEATURES].isna().sum().to_string())

In [ ]:
# Cap (winsorize) extreme values instead of deleting restaurants – we only have ~100 of them.
def cap_outliers(df, cols):
    df = df.copy()
    for c in cols:
        q1, q3 = df[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        df[c] = df[c].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return df

X_capped = cap_outliers(restaurants[CLUSTER_FEATURES], CLUSTER_FEATURES)

# Correlations between features (highly correlated features count double in distance-based clustering)
plt.figure(figsize=(8, 6))
sns.heatmap(X_capped.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between clustering features")
plt.show()

## 9. Clustering
**Why scaling?** K-Means and hierarchical clustering use distances. `Cost` is in hundreds of rupees while ratings are 1–5, so without scaling, cost would dominate. `StandardScaler` puts all features on the same scale.

**Algorithms compared:** K-Means (fast, easy to explain) and Agglomerative/Ward hierarchical clustering (gives a dendrogram).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_capped)
print("Scaled matrix:", X_scaled.shape)

### 9.1 How many clusters? Elbow, silhouette and dendrogram

In [ ]:
k_range = range(2, 11)
inertias, sil_scores = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(list(k_range), inertias, "o-")
axes[0].set_title("Elbow method (inertia)"); axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[1].plot(list(k_range), sil_scores, "o-", color="darkorange")
axes[1].set_title("Silhouette score (higher is better)"); axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.show()

print(pd.DataFrame({"k": list(k_range), "inertia": np.round(inertias, 1), "silhouette": np.round(sil_scores, 3)}).to_string(index=False))

In [ ]:
Z = linkage(X_scaled, method="ward")
plt.figure(figsize=(13, 5))
dendrogram(Z, truncate_mode="lastp", p=20, leaf_rotation=90, show_contracted=True)
plt.title("Dendrogram (Ward linkage, last 20 merges)")
plt.xlabel("Restaurants (or groups of restaurants)"); plt.ylabel("Distance")
plt.show()

In [ ]:
# Pick k. Rule: look at 3-6 clusters (2 is too coarse to be useful), find the best silhouette,
# then take the SMALLEST k whose silhouette is within 0.01 of the best -> simpler to explain.
# Change K_FINAL by hand if the elbow/dendrogram convince you otherwise.
candidates = {k: s for k, s in zip(k_range, sil_scores) if 3 <= k <= 6}
best_sil = max(candidates.values())
K_FINAL = min(k for k, s in candidates.items() if s >= best_sil - 0.01)
print("Chosen number of clusters:", K_FINAL)

**Reading the plots honestly:** silhouette scores for this data are low-to-moderate (roughly 0.2–0.3), which means the restaurants form *overlapping segments* rather than perfectly separated groups. That is normal for real business data. The clusters are still useful as segments to describe and act on, but say so in your conclusion instead of claiming they are sharply separated.

### 9.2 Fit the models and compare

In [ ]:
kmeans = KMeans(n_clusters=K_FINAL, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
agglo = AgglomerativeClustering(n_clusters=K_FINAL, linkage="ward").fit(X_scaled)

def cluster_metrics(X, labels):
    return {"silhouette (higher=better)": silhouette_score(X, labels),
            "Davies-Bouldin (lower=better)": davies_bouldin_score(X, labels),
            "Calinski-Harabasz (higher=better)": calinski_harabasz_score(X, labels)}

comparison = pd.DataFrame({"K-Means": cluster_metrics(X_scaled, kmeans.labels_),
                           "Agglomerative (Ward)": cluster_metrics(X_scaled, agglo.labels_)}).round(3)
display(comparison)

restaurants["cluster"] = kmeans.labels_
print("\nRestaurants per cluster (K-Means):")
print(restaurants["cluster"].value_counts().sort_index().to_string())

In [ ]:
# Visualize the clusters in 2D using PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
plt.figure(figsize=(9, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=restaurants["cluster"], palette="deep", s=80)
plt.title(f"K-Means clusters (k={K_FINAL}) projected to 2D with PCA")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% of variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% of variance)")
plt.legend(title="Cluster")
plt.show()

## 10. Cluster profiles & business strategies
**Explainability:** what makes each cluster different, and how can Zomato and customers use it?

In [ ]:
profile = restaurants.groupby("cluster")[CLUSTER_FEATURES].mean().round(2)
profile.insert(0, "n_restaurants", restaurants["cluster"].value_counts().sort_index())
display(profile)

# Heatmap of how each cluster differs from the overall average (z-scores)
z = (profile[CLUSTER_FEATURES] - restaurants[CLUSTER_FEATURES].mean()) / restaurants[CLUSTER_FEATURES].std()
plt.figure(figsize=(10, 0.9 * K_FINAL + 2))
sns.heatmap(z, annot=True, fmt=".1f", cmap="RdYlGn", center=0)
plt.title("Cluster profile vs overall average (in standard deviations)")
plt.ylabel("Cluster")
plt.show()

In [ ]:
# Cuisines and example restaurants in each cluster
cuis_cols = [c for c in restaurants.columns if c.startswith("cuis_")]
cuisine_share = restaurants.groupby("cluster")[cuis_cols].mean().mul(100).round(0)
cuisine_share.columns = [c.replace("cuis_", "").replace("_", " ") for c in cuisine_share.columns]
display(cuisine_share)

for c in sorted(restaurants["cluster"].unique()):
    top = restaurants[restaurants["cluster"] == c].nlargest(5, "avg_rating")["Name"].tolist()
    print(f"Cluster {c} – best-rated examples: {', '.join(top)}")

### Name each cluster and write a strategy
_Fill this table in using the profile table and heatmap above (this is the most important part for your video):_

| Cluster | Suggested name | What defines it | Strategy for **customers** | Strategy for **Zomato** |
|---|---|---|---|---|
| 0 | | | | |
| 1 | | | | |
| 2 | | | | |

**Ideas to get you started:** premium high-rated places can be promoted as "special occasion" picks; cheap but poorly-rated places may need quality support or a lower listing priority; highly-rated, low-cost places are "hidden gems" worth featuring in collections.

## 11. Predicting review sentiment from text (model + evaluation)
Besides clustering, we train a **supervised** model to check how well the *words* of a review predict whether it is positive or negative.

- Target: `positive` = rating ≥ 4, `negative` = rating ≤ 2 (3-star reviews are ambiguous and left out).
- Features: TF-IDF of the review text. Model: Logistic Regression (`class_weight="balanced"` because positives outnumber negatives).

In [ ]:
clf_df = reviews_text[reviews_text["Rating"] != 3].copy()
clf_df = clf_df[(clf_df["Rating"] >= 4) | (clf_df["Rating"] <= 2.5)]
clf_df["target"] = (clf_df["Rating"] >= 4).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    clf_df["Review"], clf_df["target"], test_size=0.2, stratify=clf_df["target"], random_state=RANDOM_STATE)

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_features=20000)
Xtr = tfidf.fit_transform(X_train)     # fit ONLY on training data to avoid leakage
Xte = tfidf.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
model.fit(Xtr, y_train)
pred = model.predict(Xte)

print("Train size:", Xtr.shape[0], "| Test size:", Xte.shape[0])
print(f"Accuracy : {accuracy_score(y_test, pred):.3f}")
print(f"Precision: {precision_score(y_test, pred):.3f}")
print(f"Recall   : {recall_score(y_test, pred):.3f}")
print(f"F1-score : {f1_score(y_test, pred):.3f}")
print()
print(classification_report(y_test, pred, target_names=["Negative", "Positive"]))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Negative", "Positive"]).plot(cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.show()

In [ ]:
# Which words push a review towards positive / negative?
words = np.array(tfidf.get_feature_names_out())
coefs = model.coef_[0]
top_pos = words[np.argsort(coefs)[-12:][::-1]]
top_neg = words[np.argsort(coefs)[:12]]
print("Most 'positive' words:", ", ".join(top_pos))
print("Most 'negative' words:", ", ".join(top_neg))

**Observation:** _(comment on accuracy vs F1 – with imbalanced data, F1 and recall for the negative class matter more than accuracy alone.)_

## 12. Conclusion
_(Write this in your own words – the numbers are in the outputs above.)_

1. **Data & cleaning:** what you fixed (cost text, rating junk value, duplicates, metadata split) and how many restaurants you ended with.
2. **EDA findings:** ratings skew high; most popular cuisines; how cost relates to rating.
3. **Sentiment:** share of positive/neutral/negative reviews, and how well text sentiment matches star ratings.
4. **Clusters:** how many segments you found, why you chose that number (elbow + silhouette + dendrogram), and what each segment means.
5. **Business impact:** how customers can use the segments to find the right restaurant, and where Zomato should focus.
6. **Limitations:** only ~100 restaurants, one city, reviews from 2016–2019, VADER is a general-purpose tool and may miss food-specific language.
7. **Next steps:** try DBSCAN / Gaussian Mixture, add location data, use a transformer-based sentiment model.

In [ ]:
# Save the final restaurant table with its cluster labels (and download it in Colab)
out_cols = ["Name", "Cost", "Cuisines", "avg_rating", "avg_sentiment", "pct_negative", "cluster"]
restaurants[out_cols].to_csv("zomato_restaurant_clusters.csv", index=False)
print("Saved zomato_restaurant_clusters.csv")
# from google.colab import files; files.download("zomato_restaurant_clusters.csv")